In [ ]:
import os, numpy as np, pandas as pd, psycopg, matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)
def q(sql, **params): return pd.read_sql(sql, conn, params=params)

## Finding 1: playtime and positivity

Full year, 2025-10-05 through 2026-10-13. Only games with 30+ reviews in both groups (metrics.md).

In [ ]:
END_DATE = "2026-10-14"   # exclusive: the full year, 2025-10-05 through 2026-10-13 (newest complete day)

per_game = q("""
    SELECT appid,
           count(*) FILTER (WHERE playtime_group = '2h or less')                       AS n_short,
           avg(voted_up::int) FILTER (WHERE playtime_group = '2h or less')              AS share_short,
           count(*) FILTER (WHERE playtime_group = '50h or more')                      AS n_long,
           avg(voted_up::int) FILTER (WHERE playtime_group = '50h or more')             AS share_long
    FROM fact_reviews
    WHERE review_date < %(end_date)s
    GROUP BY appid
""", end_date=END_DATE)
both = per_game[(per_game.n_short >= 30) & (per_game.n_long >= 30)].copy()
both["diff"] = both.share_long - both.share_short

In [ ]:
# Within-game difference: each game is compared with itself
d = both["diff"].to_numpy()
print(f"{len(both)} of {len(per_game)} games have 30+ reviews in both groups")
print(f"mean diff   {d.mean():+.3f}")
print(f"median diff {np.median(d):+.3f}")
print(f"diff > 0    {(d > 0).sum()} of {len(d)} games ({(d > 0).mean():.0%})")

In [ ]:
# 95% interval: resample games (the independent units), not reviews
rng = np.random.default_rng(42)
boot = d[rng.integers(0, len(d), size=(10_000, len(d)))].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean diff {d.mean():+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
# Pooled difference: all reviews in each group, ignoring games. Busy games weigh more here.
pooled = q("""
    SELECT playtime_group, count(*) AS reviews, avg(voted_up::int) AS share_positive
    FROM fact_reviews
    WHERE review_date < %(end_date)s AND playtime_group IN ('2h or less', '50h or more')
    GROUP BY playtime_group
""", end_date=END_DATE).set_index("playtime_group")
pooled_diff = pooled.share_positive["50h or more"] - pooled.share_positive["2h or less"]
print(pooled)
print(f"pooled diff {pooled_diff:+.3f} vs within-game mean {d.mean():+.3f}")

# Why they differ: the gap shrinks as games get busier, and busy games dominate the pooled 50h+ group
games = q("SELECT appid, name, size_group FROM dim_game")
both = both.merge(games, on="appid")
print(both.groupby("size_group").agg(games=("appid", "size"), mean_diff=("diff", "mean")))

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], color="#8a8985", lw=1, ls="--", zorder=1)
ax.scatter(both.share_short, both.share_long, s=36, color="#2a78d6", edgecolor="#fcfcfb", linewidth=1, zorder=2)
ax.text(0.97, 0.94, "same share in both groups", rotation=45, rotation_mode="anchor",
        ha="right", va="top", color="#52514e", fontsize=9)

# Label only the exception: the one game below the line
for _, g in both[both["diff"] <= 0].iterrows():
    ax.annotate(g["name"], (g.share_short, g.share_long), xytext=(0, -7), textcoords="offset points",
                ha="center", va="top", fontsize=9, color="#0b0b0b")

lim_lo = np.floor(min(both.share_short.min(), both.share_long.min()) * 10) / 10
ax.set(xlim=(lim_lo, 1), ylim=(lim_lo, 1), aspect="equal",
       xlabel="Share positive, reviews after 2 hours or less",
       ylabel="Share positive, reviews after 50 hours or more")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(color="#e6e5e1", lw=0.8)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color("#b5b4ae")
ax.tick_params(colors="#52514e")

ax.set_title(f"In {(d > 0).sum()} of {len(d)} games, players with 50+ hours review more positively\n"
             f"than players with 2 hours or less, by {d.mean() * 100:.0f} percentage points on average",
             loc="left", fontsize=11, color="#0b0b0b")
fig.text(0.01, 0.01, "Each dot is one game with 30+ reviews in both groups, 2025-10-05 to 2026-10-13.\n"
         "Dots above the dashed line: long-playtime reviewers are more positive.", fontsize=8, color="#52514e")
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig("../figures/finding1_playtime.png", dpi=150, bbox_inches="tight")
plt.show()

## Finding 2: sale weeks

Same within-game approach as Finding 1, at the day level: for each game, sale days (`dim_date.is_sale_day`, whole UTC
days) vs all other days, keeping only games with 30+ reviews on each side. Three differences, sale minus other:

- **Share positive.** The prediction is that sales make no real difference. It holds if the mean difference *and* its
  95% interval stay within ±2 percentage points.
- **Share of reviews at 2 hours or less** (out of reviews with playtime recorded), to see whether sales bring in new players.
- **Reviews per day**, as a sale/other ratio. Averaged as a geometric mean (mean of log ratios), so a game that doubles
  and one that halves cancel out.

Traps handled up front:

- **Autumn Sale 2026 is selection on the outcome.** The pool came from top sellers on 2026-10-05, mid-sale (L3), so these
  games' volume that week is high partly because they were picked for it. Results are shown with and without it.
  Without it, its days are dropped from both sides, not counted as "other" days.
- **Launch spikes.** 159 of 300 games have history that starts after 2025-10-05 (new releases, or capped backfills, L5).
  A launch week falls on non-sale days and makes them look busy. Each game's first 30 days are dropped. The
  version that keeps them is shown for comparison.
- **Winter Sale 2025 coincides with the holidays,** so a difference there may be seasonal rather than caused by the sale.

In [ ]:
# One row per game per day from the game's first covered review day, zero-filled so quiet days count toward per-day rates
days = q("""
    WITH span AS (
        SELECT appid, greatest(min(review_date), '2025-10-05'::date) AS first_day
        FROM fact_reviews
        WHERE review_date < %(end_date)s
        GROUP BY appid
    ),
    daily AS (
        SELECT appid, review_date AS date,
               count(*)                                              AS reviews,
               sum(voted_up::int)                                    AS positive,
               count(playtime_group)                                 AS with_playtime,
               count(*) FILTER (WHERE playtime_group = '2h or less') AS short_play
        FROM fact_reviews
        WHERE review_date < %(end_date)s
        GROUP BY appid, review_date
    )
    SELECT s.appid, d.date, d.is_sale_day, d.sale_name, d.date - s.first_day AS day_number,
           coalesce(r.reviews, 0) AS reviews, coalesce(r.positive, 0) AS positive,
           coalesce(r.with_playtime, 0) AS with_playtime, coalesce(r.short_play, 0) AS short_play
    FROM span s
    JOIN dim_date d ON d.date >= s.first_day AND d.date < %(end_date)s
    LEFT JOIN daily r ON r.appid = s.appid AND r.date = d.date
""", end_date=END_DATE)
print(days.groupby("sale_name").date.agg(first="min", last="max", days="nunique"))

In [ ]:
def sale_vs_other(df):
    """Per game: sale-day minus other-day differences, for games with 30+ reviews on each side."""
    cols = ["reviews", "positive", "with_playtime", "short_play"]
    g = df.groupby(["appid", "is_sale_day"])[cols].sum()
    g["days"] = df.groupby(["appid", "is_sale_day"]).size()
    w = g.unstack("is_sale_day")
    w = w[(w["reviews"][True] >= 30) & (w["reviews"][False] >= 30)]
    share_pos = w["positive"] / w["reviews"]
    share_short = w["short_play"] / w["with_playtime"]
    per_day = w["reviews"] / w["days"]
    return pd.DataFrame({"diff_positive": share_pos[True] - share_pos[False],
                         "diff_short_play": share_short[True] - share_short[False],
                         "log_ratio_per_day": np.log(per_day[True] / per_day[False])})

def summarize(per_game, label):
    """Mean of each difference with a 95% interval, resampling games."""
    rng = np.random.default_rng(42)
    row = {"version": label, "games": len(per_game)}
    for col in per_game.columns:
        x = per_game[col].dropna().to_numpy()
        boot = x[rng.integers(0, len(x), size=(10_000, len(x)))].mean(axis=1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        if col == "log_ratio_per_day":   # back to a ratio: geometric mean
            row["per_day_ratio"] = f"{np.exp(x.mean()):.2f} [{np.exp(lo):.2f}, {np.exp(hi):.2f}]"
        else:
            row[col] = f"{x.mean() * 100:+.1f} [{lo * 100:+.1f}, {hi * 100:+.1f}]"
            row[col + "_ci"] = (lo, hi)
    return row

no_launch = days[days.day_number >= 30]
versions = {
    "main: no launch month, without Autumn 2026": no_launch[no_launch.sale_name.ne("Autumn Sale 2026")],
    "no launch month, with Autumn 2026":           no_launch,
    "keep launch month, without Autumn 2026":      days[days.sale_name.ne("Autumn Sale 2026")],
    "keep launch month, with Autumn 2026":         days,
}
rows = [summarize(sale_vs_other(df), label) for label, df in versions.items()]
results = pd.DataFrame(rows).set_index("version")
print("Differences in percentage points (sale minus other), 95% CI; per-day ratio is sale / other")
print(results[["games", "diff_positive", "diff_short_play", "per_day_ratio"]].to_string())

lo, hi = rows[0]["diff_positive_ci"]
verdict = "holds" if -0.02 <= lo and hi <= 0.02 else "does not hold"
print(f"\nPrediction (share positive within ±2 points), main version: CI [{lo * 100:+.1f}, {hi * 100:+.1f}] -> {verdict}")

In [ ]:
# Each sale on its own vs all non-sale days (launch months dropped). Winter overlaps the holidays; Autumn 2026 is selected on.
sales = ["Winter Sale 2025", "Spring Sale 2026", "Summer Sale 2026", "Autumn Sale 2026"]
by_sale = pd.DataFrame([
    summarize(sale_vs_other(no_launch[no_launch.sale_name.isna() | no_launch.sale_name.eq(s)]), s) for s in sales
]).set_index("version")
print(by_sale[["games", "diff_positive", "diff_short_play", "per_day_ratio"]].to_string())

In [ ]:
# Main version plus each sale on its own: mean within-game difference with a 95% interval (resampling games)
def boot_ci(x, transform=lambda v: v):
    rng = np.random.default_rng(42)
    boot = x[rng.integers(0, len(x), size=(10_000, len(x)))].mean(axis=1)
    return [transform(v) for v in (x.mean(), *np.percentile(boot, [2.5, 97.5]))]

groups = {"All sales (main version)": versions["main: no launch month, without Autumn 2026"]}
groups |= {s: no_launch[no_launch.sale_name.isna() | no_launch.sale_name.eq(s)] for s in sales}
plot_rows = []
for label, df in groups.items():
    pg = sale_vs_other(df)
    plot_rows.append({"label": label, "games": len(pg),
                      "pos": boot_ci(pg.diff_positive.to_numpy(), lambda v: v * 100),
                      "vol": boot_ci(pg.log_ratio_per_day.to_numpy(), np.exp)})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
ys = np.arange(len(plot_rows))[::-1]
for ax, key in [(ax1, "pos"), (ax2, "vol")]:
    for y, r in zip(ys, plot_rows):
        mean, lo, hi = r[key]
        color = "#2a78d6" if y == ys[0] else "#8a8985"
        ax.plot([lo, hi], [y, y], color=color, lw=2, solid_capstyle="round")
        ax.scatter([mean], [y], s=50, color=color, edgecolor="#fcfcfb", linewidth=1.5, zorder=3)
    ax.grid(axis="x", color="#e6e5e1", lw=0.8)
    ax.set_axisbelow(True)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#b5b4ae")
    ax.tick_params(colors="#52514e", left=False)

ax1.axvspan(-2, 2, color="#2a78d6", alpha=0.07, lw=0)
ax1.axvline(0, color="#8a8985", lw=1, ls="--")
ax1.text(1.9, ys[-1] - 0.45, "prediction: within ±2 points", ha="right", va="center", fontsize=8, color="#52514e")
ax1.set(xlim=(-4, 4), xlabel="Share positive, sale minus other days (points)")
ax2.axvline(1, color="#8a8985", lw=1, ls="--")
ax2.set(xlabel="Reviews per day, sale ÷ other days")
ax2.xaxis.set_major_formatter(lambda v, _: f"{v:.1f}×")
ax1.set_yticks(ys, [f"{r['label']}  ({r['games']} games)" for r in plot_rows])
ax1.set_ylim(ys[-1] - 0.8, ys[0] + 0.5)

ax1.set_title("Sales don't change how positive reviews are...", loc="left", fontsize=11, color="#0b0b0b")
ax2.set_title("...but most bring more reviews per day", loc="left", fontsize=11, color="#0b0b0b")
fig.text(0.01, 0.01, "Within-game, games with 30+ reviews on each side; each game's first 30 days dropped. Lines are 95% intervals "
         "(resampling games).\nWinter 2025 overlaps the holidays; Autumn 2026 volume is unreliable (games selected mid-sale, L3).",
         fontsize=8, color="#52514e")
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig("../figures/finding2_sales.png", dpi=150, bbox_inches="tight")
plt.show()

**Caveats**

- **Winter Sale 2025 (Dec 18 – Jan 5) is also the holidays.** People get new PCs and gift cards and have time off,
  so its volume and new-player share may be seasonal. Excluding it from the main version is a reasonable check.
  Without a non-sale holiday period in the data, the two can't be separated.
- **Autumn Sale 2026** is shown, but selection on the outcome makes its volume ratio unreliable.
- **Reviews lag purchases.** Someone who buys on the last sale day reviews days later, on a non-sale day. Sale-day
  counts miss part of the sale's effect, so the volume ratio is a lower bound. The days after a sale could be tested
  as their own group.
- **Sale days are whole UTC days** but sales start at 10 a.m. Pacific, so each sale's first flagged day is only about
  7 hours of sale. Autumn Sale 2025 is only its last two days (Oct 5–6).
- **Sitewide sales only.** Games also go on sale individually, and those days count as "other" here. That makes the
  two groups more alike than they really are.

## Finding 3: review length and positivity

Are longer reviews more negative? If they are, week 3's text work (which skips reviews under 20 characters, D18)
will over-represent complaints.

`fact_reviews.review_chars` is the length after the same trim as `is_short`, so the first bucket is exactly the
short reviews. Four buckets: under 20, 20–199, 200–999 and 1,000+ characters. Full year, as in Finding 1.

- **Pooled:** all reviews in each bucket. Busy games weigh more.
- **Within-game:** share positive per game per bucket, averaged with each game counting once. Only games with 30+
  reviews in every bucket.

In [ ]:
BUCKETS = ["under 20", "20-199", "200-999", "1,000+"]

by_bucket = q("""
    SELECT appid,
           CASE WHEN review_chars < 20   THEN 'under 20'
                WHEN review_chars < 200  THEN '20-199'
                WHEN review_chars < 1000 THEN '200-999'
                ELSE '1,000+' END        AS bucket,
           count(*)                      AS reviews,
           sum(voted_up::int)            AS positive
    FROM fact_reviews
    WHERE review_date < %(end_date)s
    GROUP BY 1, 2
""", end_date=END_DATE)

# Pooled: every review counts once
pooled_len = by_bucket.groupby("bucket")[["reviews", "positive"]].sum().reindex(BUCKETS)
pooled_len["share_positive"] = pooled_len.positive / pooled_len.reviews
pooled_len["share_of_reviews"] = pooled_len.reviews / pooled_len.reviews.sum()

# Within-game: one share per game per bucket, games with 30+ reviews in every bucket
wide = by_bucket.pivot(index="appid", columns="bucket", values=["reviews", "positive"]).fillna(0)
keep = (wide["reviews"][BUCKETS] >= 30).all(axis=1)
share_by_game = (wide["positive"] / wide["reviews"])[BUCKETS][keep]

summary = pooled_len[["reviews", "share_of_reviews", "share_positive"]].assign(
    within_game_mean=share_by_game.mean())
print(f"{keep.sum()} of {len(wide)} games have 30+ reviews in every bucket")
print(summary.to_string(formatters={"share_of_reviews": "{:.1%}".format, "share_positive": "{:.1%}".format,
                                    "within_game_mean": "{:.1%}".format}))

In [ ]:
# Each bucket vs 20-199 within each game, 95% interval resampling games
rng = np.random.default_rng(42)
for b in ["under 20", "200-999", "1,000+"]:
    x = (share_by_game[b] - share_by_game["20-199"]).to_numpy()
    boot = x[rng.integers(0, len(x), size=(10_000, len(x)))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{b:>8} minus 20-199: {x.mean() * 100:+.1f} points [{lo * 100:+.1f}, {hi * 100:+.1f}], "
          f"lower in {(x < 0).sum()} of {len(x)} games")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
xs = np.arange(len(BUCKETS))
ax.plot(xs, pooled_len.share_positive, marker="o", color="#8a8985", lw=2, label="Pooled (all reviews)")
ax.plot(xs, share_by_game.mean(), marker="o", color="#2a78d6", lw=2, label="Within-game (each game counts once)")
for x, y in zip(xs, share_by_game.mean()):
    ax.annotate(f"{y:.0%}", (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=9, color="#0b0b0b")

ax.set_xticks(xs, [f"{b}\n{s:.0%} of reviews" for b, s in zip(BUCKETS, pooled_len.share_of_reviews)])
ax.set(xlabel="Review length (characters, trimmed)", ylabel="Share positive")
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(axis="y", color="#e6e5e1", lw=0.8)
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color("#b5b4ae")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, fontsize=9, loc="lower left")

ax.set_title("Share of positive reviews by review length", loc="left", fontsize=11, color="#0b0b0b")
fig.text(0.01, 0.01, f"Within-game: {len(share_by_game)} games with 30+ reviews in every bucket, 2025-10-05 to 2026-10-13.\n"
         "Under 20 = is_short, skipped by week 3 text work.", fontsize=8, color="#52514e")
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig("../figures/finding3_length.png", dpi=150, bbox_inches="tight")
plt.show()